# dMRI preprocessing QC - single-subject demo

This notebook shows quality-control (QC) metrics and visualizations for **one subject**
from the preprocessing stage (`den_gr`, `topup`, `eddy`, `bias`).

It does not run FSL/MRtrix/ANTs: it **only reads** outputs that those steps already
wrote to disk and turns them into tables (`pandas`) and figures (`matplotlib`) using
the `preprocessing_qc` subpackage.

**Flow:** acquisition shells -> denoising QC (SNR + before/after) -> eddy QC
(motion, outliers) -> `eddy_quad` summary (CNR) -> subject summary table ->
cohort summary (multiple subjects) -> CSV.


## 1. Configuration

Edit `BASE` and `SUB`. The notebook resolves scanner-named raw DWI sidecars and script-generated derivative names.

In [ ]:
import sys
from pathlib import Path

# Allow imports whether the notebook is launched from notebooks/ or the repo root.
for cand in [Path.cwd() / "src", Path.cwd(), Path.cwd().parent / "src", Path.cwd().parent]:
    if (cand / "preprocessing_qc").is_dir():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import preprocessing_qc as qc

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


In [ ]:
# --- EDIT THIS ---
BASE = Path.cwd() / "data"
if not BASE.exists() and (Path.cwd().parent / "data").exists():
    BASE = Path.cwd().parent / "data"
SUB = "brain3_hz000_d55_b2000"
SESSION = "ses-T0"

def one_match(folder, pattern, label):
    matches = sorted(folder.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {label} matching {pattern!r} in {folder}, found {len(matches)}")
    return matches[0]

dwi_dir  = BASE / SUB / SESSION / "dwi"
den_dir  = dwi_dir / "den"
eddy_out = den_dir / "preproc-2" / "eddy-out"
topup_out = den_dir / "preproc-1" / "topup-out"

# Raw acquisition files can keep their scanner names; derivatives use the script-generated names.
raw_dwi     = one_match(dwi_dir, "*.nii.gz", "raw DWI")
bval        = raw_dwi.with_suffix("").with_suffix(".bval")
den_dwi     = den_dir / f"{SUB}_{SESSION}_dwi_den.nii.gz"
den_grc     = den_dir / f"{SUB}_{SESSION}_dwi_den_grc.nii.gz"
noise_map   = den_dir / "denoising-der" / f"{SUB}_{SESSION}_dwi_noise.nii.gz"
residuals   = den_dir / "denoising-der" / f"{SUB}_{SESSION}_dwi_den-residuals.nii.gz"
mask        = topup_out / f"{SUB}_{SESSION}_dwi_topup-out-preproc-1-unwarp-imgs-mean_brain-f0.3R_mask.nii.gz"

# Eddy basename (without suffix): .eddy_movement_rms, .qc/qc.json, etc. hang from here.
eddy_base   = eddy_out / f"{SUB}_{SESSION}_dwi_den_grc_tec_preproc-2"
corrected   = Path(str(eddy_base) + ".nii.gz")
eddy_required = {
    "movement_rms": Path(str(eddy_base) + ".eddy_movement_rms"),
    "parameters": Path(str(eddy_base) + ".eddy_parameters"),
    "outlier_map": Path(str(eddy_base) + ".eddy_outlier_map"),
    "corrected": corrected,
    "qc_json": Path(str(eddy_base) + ".qc/qc.json"),
}
eddy_ready = all(path.exists() for path in eddy_required.values())

for label, path in {
    "raw_dwi": raw_dwi, "bval": bval, "den_dwi": den_dwi, "den_grc": den_grc,
    "noise_map": noise_map, "residuals": residuals, "mask": mask,
}.items():
    print(f"{label:10s}: {path} {'OK' if path.exists() else 'MISSING'}")
print("eddy_base :", eddy_base)
for label, path in eddy_required.items():
    print(f"eddy {label:8s}: {'OK' if path.exists() else 'MISSING'}")
if not eddy_ready:
    print("Eddy QC cells will be skipped until eddy finishes successfully.")


## 2. Acquisition Scheme (Shells)

How many directions exist for each b-value. Useful for checking that the protocol is the expected one.

In [ ]:
shells = qc.summarize_shells(bval)
display(shells)
qc.plot_shells(shells)
plt.show()


## 3. Denoising QC

b=0 SNR using the `dwidenoise` noise map (`sigma`), plus a visual before/after comparison. The *Difference* image should show noise without anatomical structure.

In [ ]:
snr = qc.compute_denoising_snr(raw_dwi, noise_map, bval, mask_path=mask)
display(snr.to_frame("value"))

qc.plot_before_after_slice(raw_dwi, den_dwi, titles=("Raw", "Denoised"), volume=0)
plt.show()


In [ ]:
# Residuals: mean ~ 0 and no structure => good denoising.
res = qc.compute_residual_stats(residuals, mask_path=mask)
display(res.to_frame("value"))


## 4. Eddy QC: Motion and Outliers

Motion RMS by volume, translation/rotation parameters, and the map of slices marked as outliers by `--repol`.

In [ ]:
if eddy_required["movement_rms"].exists():
    mrms = qc.load_movement_rms(eddy_required["movement_rms"])
    display(mrms.describe().loc[["mean", "max"]])
    qc.plot_movement_rms(mrms)
    plt.show()
else:
    print(f"Skipping: missing {eddy_required['movement_rms']}")


In [ ]:
if eddy_required["parameters"].exists():
    motion = qc.load_motion_parameters(eddy_required["parameters"])
    qc.plot_motion_parameters(motion)
    plt.show()
else:
    print(f"Skipping: missing {eddy_required['parameters']}")


In [ ]:
if eddy_required["outlier_map"].exists():
    omap = qc.load_outlier_map(eddy_required["outlier_map"])
    display(qc.summarize_outliers(omap).to_frame("value"))
    qc.plot_outlier_heatmap(omap)
    plt.show()
else:
    print(f"Skipping: missing {eddy_required['outlier_map']}")


### Eddy Before/After

Comparison between the eddy input (`den_grc`) and corrected output on one diffusion-weighted volume.

In [ ]:
if eddy_required["corrected"].exists():
    qc.plot_before_after_slice(den_grc, corrected, titles=("den_grc", "eddy-corr"), volume=30)
    plt.show()
else:
    print(f"Skipping: missing {eddy_required['corrected']}")


## 5. `eddy_quad` Summary (qc.json)

Metrics summarized by `eddy_quad`: mean motion, outlier percentage, and CNR by shell.

In [ ]:
if eddy_required["qc_json"].exists():
    qcjson = qc.load_qc_json(eddy_required["qc_json"])
    display(qc.qc_json_to_series(qcjson).to_frame("value"))
    qc.plot_cnr_per_shell(qcjson)
    plt.show()
else:
    print(f"Skipping: missing {eddy_required['qc_json']}")


## 6. Subject Summary Table

Everything above condensed into **one `pandas` row**, the unit that is later stacked across the cohort.

In [ ]:
summary = qc.subject_qc_summary(
    SUB, eddy_base,
    dwi_path=raw_dwi, noise_path=noise_map, residuals_path=residuals,
    bval_path=bval, mask_path=mask,
)
summary.T


## 7. Cohort Summary (Multiple Subjects) -> CSV

The pipeline processes **all** images in a data folder. Here we build a table
with one subject per row by scanning that folder, then compare one metric across
subjects to detect atypical cases. Edit the `SUBJECTS` list.

In [ ]:
SUBJECTS = [SUB]  # e.g. ["brain3_hz000_d55_b2000", ...]

specs = []
for s in SUBJECTS:
    dv = BASE / s / SESSION / "dwi"
    dn = dv / "den"
    raw = one_match(dv, "*.nii.gz", f"raw DWI for {s}")
    eb = dn / "preproc-2" / "eddy-out" / f"{s}_{SESSION}_dwi_den_grc_tec_preproc-2"
    specs.append(dict(
        subject_id=s, eddy_base=eb,
        dwi_path=raw,
        noise_path=dn / "denoising-der" / f"{s}_{SESSION}_dwi_noise.nii.gz",
        bval_path=raw.with_suffix("").with_suffix(".bval"),
    ))

cohort = qc.cohort_qc_summary(specs)
display(cohort)

# Between-subject comparison. Choose the column you want to audit.
if len(cohort) and "mot_abs_mm_mean" in cohort.columns:
    qc.plot_cohort_metric(cohort, "mot_abs_mm_mean", ylabel="mean absolute motion (mm)")
    plt.show()

cohort.to_csv("preprocessing_qc_summary.csv")
print("saved -> preprocessing_qc_summary.csv")


---
### AI Use Note

Parts of this notebook and the `preprocessing_qc` package were developed with AI
assistance for code and documentation refinement. Metric design, result
interpretation, and validation on data remain the author's responsibility.
